# Brain Tumor MRI Classification — DANN + ResNet-18
### v5 — Fixed & Complete (BrainMRIDataset, Criteria, Full CV Loop)

This notebook implements a **Domain-Adversarial Neural Network (DANN)** using a fine-tuned ResNet-18 for binary brain MRI classification (Tumor vs. No-Tumor).  
It trains on **BRISC-2025** (source domain) while aligning feature distributions with **Mendeley Brain MRI** (unlabelled target domain).

### Fixes in this version
| # | Issue | Fix |
|---|-------|-----|
| 1 | `BrainMRIDataset` class missing | Added in §4.1 with full docstring |
| 2 | `build_transforms()` missing | Added in §4.1 (augmented + val pipelines) |
| 3 | `CLASS_CRITERION` / `DOMAIN_CRITERION` undefined | Defined at top of §7 |
| 4 | Domain soft labels hardcoded to 0/1 | `DOMAIN_SMOOTH_SRC=0.1`, `DOMAIN_SMOOTH_TGT=0.9` |
| 5 | `run_dann_cv()` truncated — no epoch loop | Complete loop: scheduler, scaler, checkpointing, end_fold |
| 6 | `NameError` in `main()` at runtime | All dependencies defined before use |

---
# 1. Environment Setup & Imports <a id='1'></a>

In [3]:
# ==========================================
# 1. Environment Setup & Dependencies
# ==========================================
%pip install kagglehub imagehash 

In [ ]:
# ==========================================
# 1. Unified Imports & Reproducibility
# ==========================================
import os
import re
import sys
import copy
import json
import math
import pickle
import random
import logging
import itertools
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import imagehash
import kagglehub

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

# --- Constants & Seeds ---
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Logging ---
out_dir = "/content/Research_Output"
os.makedirs(out_dir, exist_ok=True)
log_file_path = os.path.join(out_dir, "pipeline_history.log")

root_logger = logging.getLogger()
if root_logger.hasHandlers():
    root_logger.handlers.clear()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    handlers=[
        logging.FileHandler(log_file_path, mode='a'), 
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger(__name__)
log.info("Pipeline logging successfully initialized and attached to file.")

In [2]:
# ==========================================
# 3. Data Acquisition & Folder Structuring
# ==========================================
log.info("Downloading datasets via Kaggle and Mendeley...")

# 1. BRISC (Source)
brisc_path = kagglehub.dataset_download("briscdataset/brisc2025")
os.makedirs("/content/DANN/source", exist_ok=True)
os.system(f"cp -r {brisc_path}/brisc2025/classification_task/train /content/DANN/source/")
os.system(f"cp -r {brisc_path}/brisc2025/classification_task/test /content/DANN/source/")

# 2. Mendeley (Target)
os.system("wget -q https://data.mendeley.com/public-api/zip/zwr4ntf94j/download/5 -O target.zip")
os.system("unzip -o -q target.zip -d /content/raw_target/")
os.system("unzip -o -q '/content/raw_target/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui/Epic and CSCR hospital Dataset.zip' -d '/content/raw_target/'")
os.makedirs("/content/DANN/target", exist_ok=True)
os.system("mv '/content/raw_target/Epic and CSCR hospital Dataset/Train' /content/DANN/target/train 2>/dev/null")
os.system("mv '/content/raw_target/Epic and CSCR hospital Dataset/Test' /content/DANN/target/test 2>/dev/null")

# 3. External Val 1 (Ayesha)
os.system("wget -q https://data.mendeley.com/public-api/zip/w4sw3s9f59/download/1 -O ext1.zip")
os.system("unzip -o -q ext1.zip -d /content/raw_ext1/")
os.system("unzip -o -q '/content/raw_ext1/Brain Tumor Data/Brain Tumor data.zip' -d /content/raw_ext1/")
os.makedirs("/content/External Validation/External-validation-dataset-1", exist_ok=True)
os.system("mv '/content/raw_ext1/Brain Tumor data/Training' '/content/External Validation/External-validation-dataset-1/train' 2>/dev/null")
os.system("mv '/content/raw_ext1/Brain Tumor data/Testing' '/content/External Validation/External-validation-dataset-1/test' 2>/dev/null")

# 4. External Val 2 (Alam)
ext2_path = kagglehub.dataset_download("alamshihab075/brain-tumor-mri-dataset-for-deep-learning")
os.makedirs("/content/External Validation/External-validation-dataset-2", exist_ok=True)
os.system(f"cp -r '{ext2_path}/Train/Train' '/content/External Validation/External-validation-dataset-2/train'")
os.system(f"cp -r '{ext2_path}/test/test' '/content/External Validation/External-validation-dataset-2/test'")

# 5. External Val 3 (DeepPy)
ext3_path = kagglehub.dataset_download("deeppythonist/brain-tumor-mri-dataset")
os.makedirs("/content/External Validation/External-validation-dataset-3", exist_ok=True)
os.system(f"cp -r '{ext3_path}/train' '/content/External Validation/External-validation-dataset-3/train'")
os.system(f"cp -r '{ext3_path}/test' '/content/External Validation/External-validation-dataset-3/test'")

log.info("All datasets acquired and structured.")

Using Colab cache for faster access to the 'brisc2025' dataset.
Using Colab cache for faster access to the 'brain-tumor-mri-dataset-for-deep-learning' dataset.
Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.


In [4]:
# ==========================================
# 4. Global Cross-Dataset Deduplication
# ==========================================
ROOT_BASE = Path("/content/")
CLEANED_DATASETS = {
    "Source_BRISC":    {"path": ROOT_BASE / "DANN/source", "priority": 1},
    "Target_Mendeley": {"path": ROOT_BASE / "DANN/target", "priority": 2},
    "Val_Ayesha":      {"path": ROOT_BASE / "External Validation/External-validation-dataset-1", "priority": 3},
    "Val_Alam":        {"path": ROOT_BASE / "External Validation/External-validation-dataset-2", "priority": 4},
    "Val_DeepPy":      {"path": ROOT_BASE / "External Validation/External-validation-dataset-3", "priority": 5},
}

DATASETS_BY_PRIORITY = sorted(CLEANED_DATASETS.items(), key=lambda x: x[1]["priority"])
all_images = []

for name, cfg in DATASETS_BY_PRIORITY:
    if not cfg["path"].exists(): continue
    imgs = [(name, str(p.resolve())) for p in cfg["path"].rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    all_images.extend(imgs)

def compute_phash(filepath):
    try:
        with Image.open(filepath) as img: return str(imagehash.phash(img))
    except Exception: return None

log.info(f"Hashing {len(all_images)} images to hunt for data leaks...")
hash_map = defaultdict(list)

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(compute_phash, img[1]): img for img in all_images}
    for future in tqdm(as_completed(futures), total=len(all_images), desc="pHashing"):
        dataset_name, filepath = futures[future]
        h = future.result()
        if h: hash_map[h].append((dataset_name, filepath))

internal_pairs, leakage_pairs = [], []
for hash_str, entries in hash_map.items():
    ds_to_paths = defaultdict(list)
    for ds, path in entries: ds_to_paths[ds].append(path)
    
    unique_entries = [] 
    for ds, paths in ds_to_paths.items():
        unique_entries.append((ds, paths[0]))
        for dup_path in paths[1:]: internal_pairs.append((ds, dup_path))
            
    if len(unique_entries) < 2: continue 
    sorted_entries = sorted(unique_entries, key=lambda e: CLEANED_DATASETS[e[0]]["priority"])
    keeper_ds, keeper_path = sorted_entries[0]

    seen_datasets = {keeper_ds}
    for dup_ds, dup_path in sorted_entries[1:]:
        if dup_ds not in seen_datasets:
            leakage_pairs.append((keeper_ds, keeper_path, dup_ds, dup_path))
            seen_datasets.add(dup_ds)

# Execute Purge
deleted_int, deleted_ext = 0, 0
for ds, path in internal_pairs:
    if os.path.exists(path): os.remove(path); deleted_int += 1
for _, _, _, path in leakage_pairs:
    if os.path.exists(path): os.remove(path); deleted_ext += 1

log.info(f"Deduplication Complete. Purged {deleted_int} internal copies and {deleted_ext} cross-dataset leaks.")

pHashing:   0%|          | 0/28815 [00:00<?, ?it/s]

In [5]:
# ==========================================
# 5. Configuration & Caching Handlers
# ==========================================
CONFIG = {
    "num_epochs":         25,
    "batch_size":         32,
    "lr":                 1e-4,
    "weight_decay":       1e-4,
    "n_folds":            5,
    "num_workers":        2,
    "seed":               42,
    "grad_clip_norm":     1.0,
    "out_dir":            "/content/Research_Output",
    "brisc_cache_path":    "/content/brisc_cleaned_cache.pkl",
    "mendeley_cache_path": "/content/mendeley_cleaned_cache.pkl",
    "slices_per_patient": 10,
}

_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_image(image_path: str, output_size: int = 224) -> np.ndarray | None:
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None

    # Morphological Crop
    _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        all_pts = np.concatenate(contours)
        x, y, w, h = cv2.boundingRect(all_pts)
        img = img[max(0, y - 2): y + h + 2, max(0, x - 2): x + w + 2]

    # Enhance, Rescale, Convert
    img = _CLAHE.apply(img)
    f = img.astype(np.float32)
    lo, hi = f.min(), f.max()
    if hi > lo: f = (f - lo) / (hi - lo)
    img = (f * 255).astype(np.uint8)
    img = cv2.resize(img, (output_size, output_size), interpolation=cv2.INTER_CUBIC)
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

def build_cache(records: list, cache_path: str = "") -> dict:
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, "rb") as f: return pickle.load(f)
    cache = {}
    for r in tqdm(records, desc="Preprocessing to RAM", unit="img"):
        img = preprocess_image(r["path"])
        cache[r["path"]] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)
    if cache_path:
        with open(cache_path, "wb") as f: pickle.dump(cache, f)
    return cache

In [6]:
# ==========================================
# 6. Dataset Class & Domain Parsers
# ==========================================
BRISC_TYPE_MAP = {"gl": "glioma", "me": "meningioma", "pi": "pituitary", "no": "no_tumor"}
NEGATIVE_FOLDERS = {"no_tumor", "notumor", "no-tumor", "normal", "healthy", "negative"}
TUMOR_KEYWORDS = {"glioma", "meningioma", "pituitary", "tumor", "yes"}
_BRISC_RE = re.compile(r'^brisc2025_(train|test)_(\d{5})_([a-z]{2})_(?:ax|co|sa)_t1', re.IGNORECASE)

def load_brisc(brisc_root: str, slices_per_patient: int = 10) -> list:
    records = []
    for dirpath, _, filenames in os.walk(brisc_root):
        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS: continue
            m = _BRISC_RE.match(Path(fname).stem)
            if not m: continue
            slice_id = int(m.group(2))
            tc = m.group(3).lower()
            patient_id = f"{tc}_{slice_id // slices_per_patient:04d}"
            records.append({"path": os.path.join(dirpath, fname), "label": 0 if tc == "no" else 1, "patient_id": patient_id})
    return records

def load_mendeley(root_path, name: str = "Mendeley") -> list:
    records = []
    for dirpath, _, filenames in os.walk(root_path):
        folder_name = os.path.basename(dirpath).lower()
        if any(neg in folder_name for neg in NEGATIVE_FOLDERS): label = 0
        elif any(pos in folder_name for pos in TUMOR_KEYWORDS): label = 1
        else: continue
        for fname in filenames:
            if Path(fname).suffix.lower() in IMG_EXTS:
                records.append({"path": os.path.join(dirpath, fname), "label": label})
    return records

class BrainMRIDataset(Dataset):
    def __init__(self, records: list, transform=None, cache: dict = None):
        self.records, self.transform, self.cache = records, transform, cache

    def __len__(self): return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        path = record["path"]
        img = self.cache.get(path) if self.cache else preprocess_image(path)
        if img is None: img = np.zeros((224, 224, 3), dtype=np.uint8)
        if self.transform: img = self.transform(img)
        return img, torch.tensor(record["label"], dtype=torch.long)

def build_transforms(augment: bool = False) -> T.Compose:
    imagenet_norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if augment:
        return T.Compose([
            T.ToPILImage(), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(10),
            T.ColorJitter(brightness=0.2, contrast=0.2), T.RandomAffine(0, translate=(0.05, 0.05)),
            T.ToTensor(), imagenet_norm
        ])
    return T.Compose([T.ToPILImage(), T.ToTensor(), imagenet_norm])

In [7]:
# ==========================================
# 7. Model Architecture & Loss Criteria
# ==========================================
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DANN_ResNet18(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        in_feat = backbone.fc.in_features

        self.class_classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_feat, 1))
        self.domain_classifier = nn.Sequential(
            nn.Dropout(p=0.6), nn.Linear(in_feat, 512), nn.BatchNorm1d(512),
            nn.ReLU(True), nn.Dropout(p=0.6), nn.Linear(512, 1)
        )

    def forward(self, x, alpha=None):
        feat = self.feature_extractor(x).view(x.size(0), -1)
        if alpha is not None:
            return self.class_classifier(feat), self.domain_classifier(GradientReversal.apply(feat, alpha))
        return self.class_classifier(feat)

CLASS_CRITERION = nn.BCEWithLogitsLoss()
DOMAIN_CRITERION = nn.BCEWithLogitsLoss()
DOMAIN_SMOOTH_SRC, DOMAIN_SMOOTH_TGT = 0.1, 0.9

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_sum, correct, total, all_labels, all_probs = 0.0, 0, 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.float().unsqueeze(1).to(device)
        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = CLASS_CRITERION(logits, labels)
        probs = torch.sigmoid(logits.float())
        loss_sum += loss.item() * imgs.size(0)
        correct += ((probs >= 0.5).long() == labels.long()).sum().item()
        total += imgs.size(0)
        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())
    model.train()
    return loss_sum / max(total, 1), correct / max(total, 1), np.array(all_labels), np.array(all_probs)

def compute_metrics(labels, probs):
    preds = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(labels, preds), "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0), "specificity": tn / (tn + fp + 1e-8),
        "f1": f1_score(labels, preds, zero_division=0), "auc_roc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0
    }

In [8]:
# ==========================================
# 8. Training Runners & External Validation
# ==========================================
def train_dann_epoch(model, source_loader, target_iter, optimizer, scaler, device, current_epoch, total_epochs, grad_clip_norm):
    model.train()
    loss_sum, correct_class, total = 0.0, 0, 0
    len_dl = len(source_loader)
    
    for i, (src_imgs, src_labels) in enumerate(source_loader):
        tgt_imgs, _ = next(target_iter)
        src_imgs, src_labels = src_imgs.to(device), src_labels.float().unsqueeze(1).to(device)
        tgt_imgs = tgt_imgs.to(device)

        p = float(i + current_epoch * len_dl) / (total_epochs * len_dl)
        alpha = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            src_class_logits, src_domain_logits = model(src_imgs, alpha)
            _, tgt_domain_logits = model(tgt_imgs, alpha)
            
            loss_cls = CLASS_CRITERION(src_class_logits, src_labels)
            loss_dom = DOMAIN_CRITERION(src_domain_logits, torch.full_like(src_domain_logits, DOMAIN_SMOOTH_SRC)) + \
                       DOMAIN_CRITERION(tgt_domain_logits, torch.full_like(tgt_domain_logits, DOMAIN_SMOOTH_TGT))
            total_loss = loss_cls + loss_dom

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        n_src = src_imgs.size(0)
        loss_sum += total_loss.item() * n_src
        correct_class += ((torch.sigmoid(src_class_logits) >= 0.5).long() == src_labels.long()).sum().item()
        total += n_src

    return loss_sum / total, correct_class / total

def run_dann_cv(brisc_records, mendeley_records, device, config, brisc_cache=None, mendeley_cache=None):
    labels_arr = np.array([r["label"] for r in brisc_records])
    groups_arr = np.array([r["patient_id"] for r in brisc_records])
    sgkf = StratifiedGroupKFold(n_splits=config["n_folds"], shuffle=True, random_state=config["seed"])
    
    target_ds = BrainMRIDataset(mendeley_records, build_transforms(augment=True), cache=mendeley_cache)
    target_dl = DataLoader(target_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True, pin_memory=PIN_MEMORY)

    best_global_auc, best_global_model = 0.0, None

    for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(brisc_records, labels_arr, groups=groups_arr), start=1):
        log.info(f"--- Fold {fold} ---")
        train_ds = BrainMRIDataset([brisc_records[i] for i in tr_idx], build_transforms(True), brisc_cache)
        val_ds = BrainMRIDataset([brisc_records[i] for i in vl_idx], build_transforms(False), brisc_cache)
        
        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True)
        val_dl = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)
        target_iter = itertools.cycle(target_dl)

        model = DANN_ResNet18(pretrained=True).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["num_epochs"], eta_min=1e-6)
        scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

        best_fold_auc = 0.0
        
        for epoch in range(config["num_epochs"]):
            tr_loss, tr_acc = train_dann_epoch(model, train_dl, target_iter, optimizer, scaler, device, epoch, config["num_epochs"], config["grad_clip_norm"])
            vl_loss, vl_acc, vl_lbl, vl_prb = evaluate(model, val_dl, device)
            scheduler.step()
            
            vl_auc = compute_metrics(vl_lbl, vl_prb)["auc_roc"]
            if vl_auc > best_fold_auc:
                best_fold_auc = vl_auc
                if vl_auc > best_global_auc:
                    best_global_auc, best_global_model = vl_auc, copy.deepcopy(model)
            log.info(f"Ep {epoch+1:02d} | Tr Loss: {tr_loss:.4f} | Vl AUC: {vl_auc:.4f}")
            
    return best_global_model

def run_external_validation(model, datasets, device, config):
    EXT_KEYS = ["Val_Ayesha", "Val_Alam", "Val_DeepPy"]
    results = {}
    for name in EXT_KEYS:
        path = datasets[name]["path"]
        if not path.exists(): continue
        records = load_mendeley(path, name)
        if not records: continue
        
        dl = DataLoader(BrainMRIDataset(records, build_transforms(False)), batch_size=config["batch_size"], num_workers=config["num_workers"])
        _, _, labels, probs = evaluate(model, dl, device)
        results[name] = compute_metrics(labels, probs)
    return results

In [ ]:
# ==========================================
# 9. Pipeline Ignition
# ==========================================
def main():
    set_seeds(CONFIG["seed"])
    
    brisc_path = str(CLEANED_DATASETS["Source_BRISC"]["path"])
    mendeley_path = str(CLEANED_DATASETS["Target_Mendeley"]["path"])
    
    brisc_records = load_brisc(brisc_path, CONFIG["slices_per_patient"])
    mendeley_records = load_mendeley(mendeley_path)
    
    log.info("Generating Caches...")
    brisc_cache = build_cache(brisc_records, CONFIG["brisc_cache_path"])
    mendeley_cache = build_cache(mendeley_records, CONFIG["mendeley_cache_path"])
    
    log.info("Initiating DANN Cross-Validation...")
    best_model = run_dann_cv(brisc_records, mendeley_records, DEVICE, CONFIG, brisc_cache, mendeley_cache)
    
    if best_model:
        torch.save(best_model.state_dict(), os.path.join(CONFIG["out_dir"], "best_model_global.pth"))
        log.info("Running Strict External Generalization Test...")
        ext_results = run_external_validation(best_model, CLEANED_DATASETS, DEVICE, CONFIG)
        for k, v in ext_results.items(): log.info(f"External [{k}] -> AUC: {v['auc_roc']:.4f} | F1: {v['f1']:.4f}")

if __name__ == "__main__":
    main()

In [2]:
# Copy from Drive to local Colab storage
!cp "/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset.zip" /content/

# Unzip it quietly (-q)
!unzip  -o -q "/content/MRI-Dataset.zip" -d /content/ 

In [3]:
!find "/content/" -maxdepth 3 -type d

/content/
/content/.config
/content/.config/configurations
/content/.config/logs
/content/.config/logs/2026.03.23
/content/MRI-Dataset
/content/MRI-Dataset/External Validation
/content/MRI-Dataset/External Validation/External-validation-dataset-1
/content/MRI-Dataset/External Validation/External-validation-dataset-3
/content/MRI-Dataset/External Validation/External-validation-dataset-2
/content/MRI-Dataset/DANN
/content/MRI-Dataset/DANN/target
/content/MRI-Dataset/DANN/source
/content/MRI-Dataset/Research_Output
/content/drive
/content/drive/Shareddrives
/content/drive/MyDrive
/content/drive/MyDrive/Classroom
/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/B tech practicals
/content/drive/MyDrive/CN_PRAC_Hardik
/content/drive/MyDrive/Peak images
/content/drive/MyDrive/mybrain_research
/content/drive/.shortcut-targets-by-id
/content/drive/.shortcut-targets-by-id/1lW1IeBmn6HOC0L_kG4h0B5muIrKqpU9U
/content/drive/.Trash-0
/content/drive/.Trash-0/files
/content/drive/.Trash-0/

---
# 2. Hyperparameters & Configuration <a id='2'></a>
All knobs in one place so you never have to hunt through the code.

In [3]:
# ==========================================
# 2. Configuration & Cleaned Paths
# ==========================================
DRIVE_BASE = Path("/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/")
ROOT_BASE = Path("/content/MRI-Dataset/")
CONFIG = {
    "num_epochs":         25,
    "batch_size":         32,
    "lr":                 1e-4,
    "weight_decay":       1e-4,
    "n_folds":            5,
    "num_workers":        2,
    "seed":               42,
    "warmup_epochs":      5,
    "grad_clip_norm":     1.0,
    "out_dir":            str(DRIVE_BASE / "Research_Output"),
    "brisc_cache_path":    "brisc_cleaned_cache.pkl",
    "mendeley_cache_path": "mendeley_cleaned_cache.pkl",
    "slices_per_patient": 10,
}

# Define your 5-Way Dataset Structure
CLEANED_DATASETS = {
    "Source_BRISC":    ROOT_BASE / "DANN/source",
    "Target_Mendeley": ROOT_BASE / "DANN/target",
    "Val_Ayesha":      ROOT_BASE / "External Validation/External-validation-dataset-1",
    "Val_Alam":        ROOT_BASE / "External Validation/External-validation-dataset-2",
    "Val_DeepPy":      ROOT_BASE / "External Validation/External-validation-dataset-3",
}
BRISC_PATH = CLEANED_DATASETS["Source_BRISC"]
MENDELEY_PATH = CLEANED_DATASETS["Target_Mendeley"]

# Ensure all directories exist to prevent assignment errors
for path_key, path_obj in CLEANED_DATASETS.items():
    path_obj.mkdir(parents=True, exist_ok=True)
os.makedirs(CONFIG["out_dir"], exist_ok=True)

---
# 3. Data Preprocessing & Caching <a id='3'></a>

### Design philosophy
1. **Auto-crop** — find the brain boundary via the largest external contour.
2. **CLAHE** — enhance local contrast to highlight tumour margins.
3. **Min-max rescale → uint8** — produce a clean [0, 255] image for PyTorch transforms.
4. **ImageNet normalisation** — applied later inside `build_transforms`, not here.

> **Why no Z-score inside `preprocess_image`?**  
> The original code applied Z-score *then* min-max in sequence. Min-max mapping to [0, 1]  
> completely overwrites the Z-score distribution — making the Z-score step a pure CPU  
> waste. Only one rescaling method is needed here; ImageNet `T.Normalize` handles the  
> final distribution shift that ResNet-18 expects.

In [4]:
# FIX: Create the CLAHE object ONCE at module level.
# Original code called cv2.createCLAHE() inside preprocess_image(), allocating a new
# C++ object for every single image (13,000+ times). This is wasteful.
_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def preprocess_image(image_path: str, output_size: int = 224) -> np.ndarray | None:
    """
    Reads, cleans, and standardises an MRI scan.

    Pipeline:
        Grayscale load → morphological crop → CLAHE → min-max [0,255] → RGB resize

    Args:
        image_path: Path to the image file.
        output_size: Target square resolution (default 224 for ResNet-18).

    Returns:
        np.ndarray of shape (H, W, 3) dtype uint8, or None if load fails.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # ── Step 1: Auto-crop ─────────────────────────────────────
    # Isolate the brain by finding the bounding rect of all external contours.
    _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh    = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        all_pts    = np.concatenate(contours)
        x, y, w, h = cv2.boundingRect(all_pts)
        pad        = 2
        img = img[max(0, y - pad): y + h + pad, max(0, x - pad): x + w + pad]

    # ── Step 2: CLAHE ─────────────────────────────────────────
    # Enhances local contrast to highlight tumour boundaries.
    # Using the module-level _CLAHE constant (created once, reused always).
    img = _CLAHE.apply(img)

    # ── Step 3: Min-max rescale → uint8 ──────────────────────
    # FIX: Removed the redundant Z-score normalisation that preceded this step.
    # Z-score output was IMMEDIATELY overwritten by min-max, so it did nothing.
    # ImageNet T.Normalize in the transform pipeline handles distribution alignment.
    f   = img.astype(np.float32)
    lo, hi = f.min(), f.max()
    if hi > lo:                          # avoid divide-by-zero on blank images
        f = (f - lo) / (hi - lo)
    img = (f * 255).astype(np.uint8)

    # ── Step 4: Resize + convert to 3-channel RGB ─────────────
    img = cv2.resize(img, (output_size, output_size), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    return img


def build_cache(records: list, cache_path: str = "") -> dict:
    """
    Pre-processes all images and stores them in RAM (with optional pickle backup).

    Memory note: 224×224×3 uint8 ≈ 150 KB per image.
    A 13,200-image cache ≈ 1.9 GB RAM. Verify you have headroom before enabling.
    """
    if cache_path and os.path.exists(cache_path):
        log.info(f"Loading existing image cache from '{cache_path}'...")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    log.info(f"Building cache for {len(records)} images (≈{len(records)*150/1024:.0f} MB RAM)...")
    cache = {}
    for r in tqdm(records, desc="Preprocessing", unit="img"):
        img = preprocess_image(r["path"])
        # Fallback: zero image if file is corrupted / missing
        cache[r["path"]] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)

    if cache_path:
        with open(cache_path, "wb") as f:
            pickle.dump(cache, f)
        log.info(f"Cache saved to '{cache_path}'.")

    return cache


---
# 4. Dataset Handlers & DataLoaders <a id='4'></a>

### Patient grouping — why it matters and its limitations
The BRISC dataset stores slices with a global sequential slice ID. We heuristically bucket every `slices_per_patient` consecutive IDs into a single patient group so that `StratifiedGroupKFold` can prevent the same patient's slices from appearing in both train and validation sets.

> **⚠️ Limitation:** If the actual number of slices per patient is not uniform (which is typical in real MRI data), some physical patients' slices will span two adjacent buckets and therefore *can* appear in both splits — a mild data leakage. The gold-standard fix is to use a metadata CSV from the BRISC dataset if one is provided. We add a log warning so you are never silently blind to this.

### Augmentation policy
- **Train**: Horizontal/vertical flips, small rotations, brightness/contrast jitter, and subtle translations. These are all physically plausible for MRI scans.
- **Val / Inference**: Only resize + normalise — no stochastic transforms.

In [17]:
# ── Constants ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BRISC_TYPE_MAP    = {"gl": "glioma", "me": "meningioma", "pi": "pituitary", "no": "no_tumor"}
NEGATIVE_FOLDERS  = {"no_tumor", "notumor", "no-tumor", "normal", "healthy", "negative"}
_BRISC_RE         = re.compile(
    r'^brisc2025_(train|test)_(\d{5})_([a-z]{2})_(?:ax|co|sa)_t1', re.IGNORECASE
)

# ━━ BRISC Loader ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def parse_brisc_filename(filename: str, slices_per_patient: int):
    stem = Path(filename).stem
    m    = _BRISC_RE.match(stem)
    if not m: return None
    slice_id  = int(m.group(2))
    type_code = m.group(3).lower()
    label     = 0 if type_code == "no" else 1
    patient_bucket = slice_id // slices_per_patient
    patient_id     = f"{type_code}_{patient_bucket:04d}"
    return {"slice_id": slice_id, "type_code": type_code, "type_name": BRISC_TYPE_MAP.get(type_code, type_code), "label": label, "patient_id": patient_id, "split": m.group(1).lower()}

def load_brisc(brisc_root: str, slices_per_patient: int = 10) -> list:
    log.info(f"Scanning BRISC images recursively in: {brisc_root}")
    records = []
    for dirpath, _, filenames in os.walk(brisc_root):
        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS: continue
            parsed = parse_brisc_filename(fname, slices_per_patient)
            if parsed is None: continue
            records.append({"path": os.path.join(dirpath, fname), "label": parsed["label"], "patient_id": parsed["patient_id"], "type_name": parsed["type_name"], "slice_id": parsed["slice_id"]})

    if not records:
        log.error(f"No valid BRISC images found in {brisc_root}. Filename pattern mismatch?")
    else:
        log.info(f"BRISC loaded: {len(records)} images")
    return records

# ━━ Mendeley Loader ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TUMOR_KEYWORDS = {"glioma", "meningioma", "pituitary", "tumor", "yes"}

def load_mendeley(root_path, name: str = "Mendeley") -> list:
    records = []
    root_path = Path(root_path)
    if not root_path.exists():
        log.error(f"❌ {name} path not found at {root_path}")
        return []
        
    for dirpath, _, filenames in os.walk(root_path):
        folder_name = os.path.basename(dirpath).lower()
        
        # Check if folder name contains any negative or positive keywords
        is_negative = any(neg in folder_name for neg in NEGATIVE_FOLDERS)
        is_positive = any(pos in folder_name for pos in TUMOR_KEYWORDS)
        
        if is_negative:
            label = 0
        elif is_positive:
            label = 1
        else:
            continue  # Skip folders that are neither (like 'train' or 'test' parent folders)
            
        for fname in filenames:
            if Path(fname).suffix.lower() in IMG_EXTS:
                records.append({
                    "path": os.path.join(dirpath, fname), 
                    "label": label, 
                    "dataset": name
                })
                
    log.info(f"✅ {name} loaded: {len(records)} images.")
    return records

---
### 4.1  Dataset Class & Transform Builder

**`BrainMRIDataset`** bridges the raw record lists produced by the loaders above into PyTorch's `Dataset` API.  
**`build_transforms`** returns either the augmented (train) or deterministic (val) transform pipeline.

> `ToPILImage()` is always the first transform because `preprocess_image()` returns `np.ndarray` (uint8 H×W×3); PyTorch's random spatial ops require PIL input.

In [7]:
# ==========================================
# 4.1. Dataset Class & Transform Builder
# ==========================================
# WHY THIS EXISTS:
#   - BrainMRIDataset wraps a list of {path, label} records into a PyTorch Dataset.
#   - build_transforms() returns train-time (augmented) or val-time (clean) transforms.
#   - Both are required by DataLoader in the CV runner (Section 8).

class BrainMRIDataset(Dataset):
    """
    Wraps a list of records into a PyTorch Dataset for brain MRI classification.

    Args:
        records  : List of dicts, each with keys 'path' (str) and 'label' (int 0/1).
        transform: torchvision transform pipeline (from build_transforms).
        cache    : Optional dict {path -> np.ndarray} from build_cache() for fast I/O.
    """
    def __init__(self, records: list, transform=None, cache: dict = None):
        self.records   = records
        self.transform = transform
        self.cache     = cache

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        path   = record["path"]
        label  = record["label"]

        # ── Load from RAM cache first; fall back to disk ──────────────
        if self.cache is not None and path in self.cache:
            img = self.cache[path]
        else:
            img = preprocess_image(path)
            if img is None:
                # Corrupted / missing file → return a black image to avoid crashing
                img = np.zeros((224, 224, 3), dtype=np.uint8)

        # ── Apply augmentation / normalisation pipeline ────────────────
        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)


def build_transforms(augment: bool = False) -> T.Compose:
    """
    Returns an image transform pipeline for the given split.

    Train (augment=True):
        Horizontal/vertical flips + small rotation + colour jitter + affine shift.
        All ops are physically plausible for MRI (no unrealistic colour inversion etc.).

    Val / Inference (augment=False):
        Only ToTensor + ImageNet normalisation — zero stochastic transforms.

    Note: T.ToPILImage() is the first step because preprocess_image() returns
    an np.ndarray (uint8 H×W×3). PyTorch random transforms require PIL input.
    """
    imagenet_normalize = T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    if augment:
        return T.Compose([
            T.ToPILImage(),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.3),
            T.RandomRotation(degrees=10),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ToTensor(),
            imagenet_normalize,
        ])

    # Val / inference — deterministic only
    return T.Compose([
        T.ToPILImage(),
        T.ToTensor(),
        imagenet_normalize,
    ])


In [30]:
# ==========================================
# 4.5. Visualizing Domain Differences (The "Eye Test")
# ==========================================
def plot_domain_comparison(brisc_recs, mendeley_recs, num_samples=5):
    if not brisc_recs or not mendeley_recs:
        log.error("Records not loaded! Check your dataset paths.")
        return
    log.info(f"Generating comparison: BRISC vs Mendeley...")
    brisc_samples = random.sample(brisc_recs, num_samples)
    mendeley_samples = random.sample(mendeley_recs, num_samples)
    fig, axes = plt.subplots(2, num_samples, figsize=(3 * num_samples, 6))
    fig.suptitle("Domain Comparison: BRISC (Top) vs. Mendeley (Bottom)", fontsize=16)
    for i in range(num_samples):
        img_s = preprocess_image(brisc_samples[i]["path"])
        if img_s is not None: axes[0, i].imshow(img_s)
        axes[0, i].set_title("BRISC")
        axes[0, i].axis("off")
        img_t = preprocess_image(mendeley_samples[i]["path"])
        if img_t is not None: axes[1, i].imshow(img_t)
        axes[1, i].set_title("Mendeley")
        axes[1, i].axis("off")
    plt.tight_layout()
    plt.show()

---
# 5. DANN Architecture & Metrics <a id='5'></a>

## Architecture Overview

```
Input MRI (B, 3, 224, 224)
        │
        ▼
  ResNet-18 Backbone                ← Shared feature extractor (frozen head removed)
  (without final FC layer)
        │
   features (B, 512)
        │
   ┌────┴──────────────────────────┐
   │                               │
   ▼                               ▼
Class Classifier            Gradient Reversal Layer (α)
Linear(512→256)→ReLU→D       │  reverses gradient sign during backprop
→Linear(256→1)               ▼
   │                    Domain Classifier
   ▼                    Linear(512→256)→ReLU→D→Linear(256→1)
Tumour logit              │
                          ▼
                    BRISC (0) or Figshare (1) logit
```

## Gradient Reversal Layer — Correctness Verification
- **Forward**: identity (`x.view_as(x)`).
- **Backward**: `grad_output × (−α)` — multiplying by negative alpha flips the gradient sign.
- The domain classifier minimises cross-entropy (wants to tell domains apart).
- The GRL forces the **feature extractor** to *maximise* domain confusion (gradient ascent on domain loss).
- The class classifier still receives real (non-reversed) gradients → classification improves.

## Balanced Head Capacity — Fix
The original class head had **no hidden layer** (`Dropout → Linear(512,1)`), while the domain head had a full 256-unit hidden layer. This asymmetry biases the adversarial game because the domain classifier converges far faster, overwhelming the feature extractor. Both heads now share the same `Linear(512→256)→ReLU→Dropout→Linear(256→1)` structure.

### 5. Updated DANN Architecture (Hobbled Domain Head)
We are increasing the Dropout rate in the domain classifier to slow down its learning rate relative to the feature extractor. We are also allowing the GRL to accept a custom `lambda_weight`.

In [8]:
# ==========================================
# 5. DANN Model
# ==========================================
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Multiply by -alpha to reverse and scale the gradient
        output = grad_output.neg() * ctx.alpha
        return output, None

class DANN_ResNet18(nn.Module):
    def __init__(self, pretrained: bool = True):
        super(DANN_ResNet18, self).__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)

        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        in_feat = backbone.fc.in_features

        self.class_classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_feat, 1)
        )

        # INCREASED CAPACITY & DROPOUT: Makes it harder for the domain head to memorize shortcuts
        self.domain_classifier = nn.Sequential(
            nn.Dropout(p=0.6), # Increased from 0.5
            nn.Linear(in_feat, 512), # Wider bottleneck
            nn.BatchNorm1d(512),     # Added BatchNorm to stabilize adversarial gradients
            nn.ReLU(True),
            nn.Dropout(p=0.6),
            nn.Linear(512, 1)
        )

    def forward(self, x, alpha=None):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)

        class_output = self.class_classifier(features)

        if alpha is not None:
            reverse_features = GradientReversal.apply(features, alpha)
            domain_output = self.domain_classifier(reverse_features)
            return class_output, domain_output

        return class_output

---
# 6. Logging Infrastructure <a id='6'></a>

This module implements a self-contained **`TrainingLogger`** that captures every signal worth monitoring during DANN training. It writes a machine-readable JSON log alongside the human-readable text log, so you can post-process results programmatically.

### What is tracked

| Category | Signals |
|---|---|
| **Per-batch** | Total loss, class loss, domain loss, batch accuracy, alpha (α), gradient L2-norm, GPU memory used |
| **Per-epoch** | All above averaged + val loss, val accuracy, val AUC-ROC, F1, sensitivity, specificity, precision, LR, epoch wall-clock time, imgs/sec throughput |
| **Domain health** | Domain classifier accuracy per epoch — verifies domain confusion is actually happening |
| **Fold summary** | Best epoch, best AUC, all final metrics, confusion matrix breakdown |
| **CV summary** | Mean ± std across all folds for every metric |
| **Overfitting detector** | Flags when train-val accuracy gap exceeds a configurable threshold |
| **ETA** | Estimated time remaining based on exponential moving average of epoch duration |


### 6. Updated Training Loop (Label Smoothing & GRL Multiplier)
This loop introduces a `lambda_weight` to scale up the adversarial penalty. It also uses soft labels (0.1 and 0.9) instead of hard labels (0.0 and 1.0) for the domain classifier to prevent gradient vanishing.

In [9]:
# ==========================================
# 6. Rich Training Logger
# ==========================================
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from typing import Optional
import torch
# from sklearn.metrics import confusion_matrix, roc_curve

# ── ANSI colour helpers ──────
_W  = "\033[0m"; _B  = "\033[1m"; _G  = "\033[32m"; _Y  = "\033[33m"; _R  = "\033[31m"

def _fmt(v: float, decimals: int = 4) -> str: return f"{v:.{decimals}f}"
def _bar(fraction: float, width: int = 30) -> str:
    filled = int(round(width * fraction))
    return "[" + "█" * filled + "─" * (width - filled) + "]"
def _hline(char: str = "─", width: int = 70) -> str: return char * width

@dataclass
class EpochRecord:
    fold: int; epoch: int; total_epochs: int
    train_loss: float = 0.0; train_acc: float = 0.0
    class_loss: float = 0.0; domain_loss: float = 0.0
    domain_acc_src: float = 0.0; domain_acc_tgt: float = 0.0
    grad_norm: float = 0.0; alpha: float = 0.0; lr: float = 0.0
    imgs_per_sec: float = 0.0; gpu_mem_mb: float = 0.0
    val_loss: float = 0.0; val_acc: float = 0.0
    val_auc: float = 0.0; val_f1: float = 0.0
    val_sensitivity: float = 0.0; val_specificity: float = 0.0; val_precision: float = 0.0
    epoch_secs: float = 0.0; is_best: bool = False

@dataclass
class FoldRecord:
    fold: int
    best_epoch: int = 0; best_val_auc: float = 0.0
    accuracy: float = 0.0; precision: float = 0.0; recall: float = 0.0
    specificity: float = 0.0; f1: float = 0.0; auc_roc: float = 0.0
    tp: int = 0; tn: int = 0; fp: int = 0; fn: int = 0
    epochs: list = field(default_factory=list)

class TrainingLogger:
    OVERFITTING_GAP = 0.10
    EMA_ALPHA       = 0.3

    def __init__(self, out_dir: str, n_folds: int, total_epochs: int):
        self.out_dir = out_dir
        self.n_folds = n_folds
        self.total_epochs = total_epochs
        self.json_path = os.path.join(out_dir, "training_journal.json")

        self.fold_records: list[FoldRecord] = []
        self._cur_fold: Optional[FoldRecord] = None
        self._cur_epoch: Optional[EpochRecord] = None

        self._batch_losses = []; self._batch_class_losses = []; self._batch_dom_losses = []
        self._batch_grad_norms = []; self._batch_dom_correct_src = []; self._batch_dom_correct_tgt = []
        self._batch_alphas = []; self._batch_n = []

        self._ema_epoch_secs: Optional[float] = None
        self._overall_start: float = time.time()

        os.makedirs(out_dir, exist_ok=True)
        log.info(_hline("═"))
        log.info(f"  TrainingLogger initialised  │  {n_folds} folds × {total_epochs} epochs")
        log.info(f"  Journal → {self.json_path}")
        log.info(_hline("═"))

    def begin_fold(self, fold: int, n_train: int, n_val: int):
        self._cur_fold = FoldRecord(fold=fold)
        log.info(f"\n{_hline('═')}\n  {_B}FOLD {fold}/{self.n_folds}{_W}  │  Train={n_train:,}  │  Val={n_val:,}\n{_hline('═')}")

    def end_fold(self, metrics: dict, val_labels: np.ndarray, val_probs: np.ndarray):
        r = self._cur_fold
        r.accuracy, r.precision, r.recall = metrics["accuracy"], metrics["precision"], metrics["recall"]
        r.specificity, r.f1, r.auc_roc = metrics["specificity"], metrics["f1"], metrics["auc_roc"]
        preds = (val_probs >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(val_labels, preds, labels=[0,1]).ravel()
        r.tp, r.tn, r.fp, r.fn = int(tp), int(tn), int(fp), int(fn)
        self.fold_records.append(r)
        self._print_fold_summary(r, val_labels, val_probs)
        self._flush_json()

    def begin_epoch(self, fold: int, epoch: int):
        self._cur_epoch = EpochRecord(fold=fold, epoch=epoch, total_epochs=self.total_epochs)
        for lst in [self._batch_losses, self._batch_class_losses, self._batch_dom_losses, self._batch_grad_norms,
                    self._batch_dom_correct_src, self._batch_dom_correct_tgt, self._batch_alphas, self._batch_n]: lst.clear()
        self._epoch_start = time.time(); self._epoch_imgs = 0

    def log_batch(self, total_loss, class_loss, domain_loss, grad_norm, alpha, n_src, src_domain_logits, tgt_domain_logits):
        with torch.no_grad():
            dom_acc_src = (torch.sigmoid(src_domain_logits.float()) < 0.5).float().mean().item()
            dom_acc_tgt = (torch.sigmoid(tgt_domain_logits.float()) >= 0.5).float().mean().item()
        self._batch_losses.append(total_loss); self._batch_class_losses.append(class_loss); self._batch_dom_losses.append(domain_loss)
        self._batch_grad_norms.append(grad_norm); self._batch_alphas.append(alpha)
        self._batch_dom_correct_src.append(dom_acc_src); self._batch_dom_correct_tgt.append(dom_acc_tgt)
        self._batch_n.append(n_src); self._epoch_imgs += n_src

    def end_epoch(self, train_loss, train_acc, val_loss, val_acc, val_metrics, lr, is_best):
        elapsed = time.time() - self._epoch_start
        self._ema_epoch_secs = elapsed if self._ema_epoch_secs is None else (self.EMA_ALPHA * elapsed + (1 - self.EMA_ALPHA) * self._ema_epoch_secs)
        gpu_mem = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

        r = self._cur_epoch
        r.train_loss, r.train_acc, r.val_loss, r.val_acc = train_loss, train_acc, val_loss, val_acc
        r.val_auc, r.val_f1, r.lr, r.is_best, r.epoch_secs = val_metrics.get("auc_roc", 0.0), val_metrics.get("f1", 0.0), lr, is_best, elapsed
        r.val_sensitivity, r.val_specificity, r.val_precision = val_metrics.get("recall", 0.0), val_metrics.get("specificity", 0.0), val_metrics.get("precision", 0.0)
        r.imgs_per_sec, r.gpu_mem_mb = self._epoch_imgs / max(elapsed, 1e-6), gpu_mem
        r.alpha = float(np.mean(self._batch_alphas)) if self._batch_alphas else 0.0
        r.grad_norm = float(np.mean(self._batch_grad_norms)) if self._batch_grad_norms else 0.0
        r.class_loss = float(np.mean(self._batch_class_losses)) if self._batch_class_losses else 0.0
        r.domain_loss = float(np.mean(self._batch_dom_losses)) if self._batch_dom_losses else 0.0
        r.domain_acc_src = float(np.mean(self._batch_dom_correct_src)) if self._batch_dom_correct_src else 0.0
        r.domain_acc_tgt = float(np.mean(self._batch_dom_correct_tgt)) if self._batch_dom_correct_tgt else 0.0

        if self._cur_fold:
            self._cur_fold.epochs.append(asdict(r))
            if is_best: self._cur_fold.best_epoch, self._cur_fold.best_val_auc = r.epoch + 1, r.val_auc

        self._print_epoch_row(r)
        return r

    def print_epoch_header(self):
        log.info(_hline())
        log.info(f"  {'Ep':>4}  {'Progress':30}  {'TrLoss':>8}  {'TrAcc':>6}  {'VlLoss':>8}  {'VlAcc':>6}  {'AUC':>6}  {'F1':>6}  {'Sens':>6}  {'Spec':>6}  {'α':>5}  {'GNorm':>7}  {'DomSrc':>7}  {'DomTgt':>7}  {'LR':>9}  {'ImgS':>7}  {'GPU MB':>7}  {'ETA':>8}")
        log.info(_hline())

    def _print_epoch_row(self, r: EpochRecord):
        pct = ((r.epoch + 1) / r.total_epochs) * 100
        epochs_left = r.total_epochs - (r.epoch + 1)
        eta_str = f"{int((epochs_left * self._ema_epoch_secs)//60)}:{int((epochs_left * self._ema_epoch_secs)%60):02d}" if self._ema_epoch_secs and epochs_left > 0 else "  done  "
        auc_col = _G if r.val_auc >= 0.90 else (_Y if r.val_auc >= 0.80 else _R)
        dsrc_col = _G if abs(r.domain_acc_src - 0.5) < 0.15 else _Y
        dtgt_col = _G if abs(r.domain_acc_tgt - 0.5) < 0.15 else _Y
        best_tag = f" {_G}★ BEST{_W}" if r.is_best else ""

        log.info(
            f"  {r.epoch+1:>4}  {_bar(pct/100, 28)} {pct:5.1f}%  {r.train_loss:>8.4f}  {r.train_acc:>6.4f}  "
            f"{r.val_loss:>8.4f}  {r.val_acc:>6.4f}  {auc_col}{_fmt(r.val_auc)}{_W}  {_fmt(r.val_f1):>6}  "
            f"{_fmt(r.val_sensitivity):>6}  {_fmt(r.val_specificity):>6}  {r.alpha:>5.3f}  {r.grad_norm:>7.4f}  "
            f"{dsrc_col}{r.domain_acc_src:>7.4f}{_W}  {dtgt_col}{r.domain_acc_tgt:>7.4f}{_W}  "
            f"{r.lr:>9.2e}  {r.imgs_per_sec:>7.1f}  {r.gpu_mem_mb:>7.0f}  {eta_str:>8}{best_tag}"
        )
        gap = r.train_acc - r.val_acc
        if gap > self.OVERFITTING_GAP: log.warning(f"  {_Y}⚠  Overfitting signal: train_acc − val_acc = {gap:.3f}{_W}")

    def log_best_model(self, fold: int, epoch: int, auc: float, save_path: str):
        log.info(f"\n  {_G}{'━'*60}\n  {_G}  ★  NEW BEST MODEL — Fold {fold}  │  Epoch {epoch}  │  AUC = {auc:.6f}\n  {_G}{'━'*60}\n")

    def _print_fold_summary(self, r: FoldRecord, val_labels, val_probs):
        log.info(f"\n{_hline('═')}\n  {_B}FOLD {r.fold} SUMMARY{_W}  │  Best epoch: {r.best_epoch}  │  Best AUC: {r.best_val_auc:.6f}\n{_hline('─')}")
        log.info(f"  Confusion Matrix (threshold=0.5):\n           Pred No-Tumor  Pred Tumor\n  Act No-Tumor   TN={r.tn:>6}    FP={r.fp:>6}\n  Act Tumor      FN={r.fn:>6}    TP={r.tp:>6}\n{_hline('═')}")

    def print_cv_summary(self):
        if not self.fold_records: return
        log.info(f"\n{_hline('═')}\n  {_B}CROSS-VALIDATION FINAL REPORT{_W}\n{_hline('═')}")
        best_fold = max(self.fold_records, key=lambda r: r.auc_roc)
        log.info(f"  {_G}★ Best fold: Fold {best_fold.fold}  │  AUC={best_fold.auc_roc:.6f}  │  Best epoch: {best_fold.best_epoch}{_W}\n{_hline('═')}")
        self._flush_json()

    def _flush_json(self):
        # ── numpy-safe JSON serialiser ─────────────────────────────────────
        # asdict() preserves numpy scalar types (numpy.bool_, numpy.float64,
        # numpy.int64) that json.dump cannot handle natively.
        # The `default` hook catches any non-serialisable type and converts it
        # to the nearest plain-Python equivalent before raising.
        def _np_safe(obj):
            if isinstance(obj, np.bool_):    return bool(obj)
            if isinstance(obj, np.integer):  return int(obj)
            if isinstance(obj, np.floating): return float(obj)
            if isinstance(obj, np.ndarray):  return obj.tolist()
            raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

        tmp = self.json_path + ".tmp"
        with open(tmp, "w") as f:
            json.dump({"folds": [asdict(r) for r in self.fold_records]}, f, indent=2, default=_np_safe)
        os.replace(tmp, self.json_path)

---
# 7. Training & Evaluation Functions <a id='7'></a>

The training loop is now wired to the `TrainingLogger` at every granularity level:

- **Per-batch**: loss decomposition, gradient L2 norm, alpha, domain classifier accuracy
- **Per-epoch**: throughput (imgs/sec), GPU memory peak, ETA, overfitting gap detector
- **Per-fold**: ROC curve saved to disk, confusion matrix, per-metric coloured bar chart in log


In [10]:
# ==========================================
# 7. Loss Criteria, Training & Evaluation
# ==========================================
# SECTION OVERVIEW:
#   7a. Loss criteria — defined ONCE here so every function below can reference them.
#   7b. _compute_grad_norm — L2 norm of all gradients for health monitoring.
#   7c. train_dann_epoch — one DANN training epoch with adversarial + class losses.
#   7d. evaluate — inference-mode pass returning loss, accuracy, labels, probs.
#   7e. compute_metrics — full clinical metric suite (sensitivity, specificity, AUC, F1).

import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from typing import Optional

# ── 7a. Loss Criteria ──────────────────────────────────────────────────────
# BCEWithLogitsLoss = sigmoid + binary cross-entropy fused (numerically stable).
# Using label smoothing for the DOMAIN criterion (soft targets 0.1 / 0.9) to
# prevent the domain classifier from becoming over-confident, which would cause
# gradient vanishing through the GRL into the feature extractor.

CLASS_CRITERION = nn.BCEWithLogitsLoss()

# Soft-label domain criterion — replaces hard 0.0 / 1.0 with 0.1 / 0.9
# The actual soft label tensors are created in the training loop using torch.full_like.
DOMAIN_CRITERION = nn.BCEWithLogitsLoss()

DOMAIN_SMOOTH_SRC = 0.1   # Soft label for "source domain" (instead of hard 0)
DOMAIN_SMOOTH_TGT = 0.9   # Soft label for "target domain" (instead of hard 1)


# ── 7b. Gradient Norm ──────────────────────────────────────────────────────
def _compute_grad_norm(model: nn.Module) -> float:
    """
    Computes the global L2 norm of all parameter gradients.
    Called AFTER scaler.unscale_() so we see true (unscaled) gradient magnitudes.

    Healthy norm: stable and not exploding. Values > 10 with clipping = frequent clipping.
    Values near 0: vanishing gradients — check GRL alpha schedule.
    """
    total_sq = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_sq += p.grad.detach().data.norm(2).item() ** 2
    return math.sqrt(total_sq)


# ── 7c. DANN Training Epoch ────────────────────────────────────────────────
def train_dann_epoch(
    model:          DANN_ResNet18,
    source_loader:  DataLoader,
    target_iter,
    optimizer:      optim.Optimizer,
    scaler:         torch.amp.GradScaler,
    device:         torch.device,
    current_epoch:  int,
    total_epochs:   int,
    grad_clip_norm: float = 1.0,
    logger:         Optional[TrainingLogger] = None,
) -> tuple[float, float]:
    """
    One DANN training epoch — fully wired to TrainingLogger.

    Alpha schedule (GRL multiplier):
        α = 2 / (1 + exp(−10p)) − 1,  where p = global_step / total_steps
        Starts near 0 (early training: focus on classification) and
        asymptotes to 1 (late training: full domain adversarial pressure).

    Returns:
        (mean_total_loss, mean_class_accuracy)
    """
    model.train()
    loss_sum, correct_class, total = 0.0, 0, 0
    len_dataloader = len(source_loader)

    pbar = tqdm(
        enumerate(source_loader),
        total=len_dataloader,
        desc=f"  Train ep {current_epoch+1:02d}/{total_epochs}",
        leave=False,
        ncols=110,
    )

    for i, (src_imgs, src_labels) in pbar:
        tgt_imgs, _ = next(target_iter)

        src_imgs   = src_imgs.to(device, non_blocking=True)
        src_labels = src_labels.float().unsqueeze(1).to(device, non_blocking=True)
        tgt_imgs   = tgt_imgs.to(device, non_blocking=True)

        # ── Alpha schedule (GRL multiplier) ───────────────────────────────
        global_step = float(i + current_epoch * len_dataloader)
        total_steps = float(total_epochs * len_dataloader)
        p     = global_step / total_steps
        alpha = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            # ── Source: class + domain loss ────────────────────────────────
            src_class_logits, src_domain_logits = model(src_imgs, alpha)
            src_class_loss = CLASS_CRITERION(src_class_logits, src_labels)

            # Soft source domain labels (0.1 instead of hard 0)
            src_domain_labels = torch.full_like(src_domain_logits, DOMAIN_SMOOTH_SRC)
            src_domain_loss   = DOMAIN_CRITERION(src_domain_logits, src_domain_labels)

            # ── Target: domain loss only (no class labels available) ───────
            _, tgt_domain_logits = model(tgt_imgs, alpha)
            tgt_domain_labels    = torch.full_like(tgt_domain_logits, DOMAIN_SMOOTH_TGT)
            tgt_domain_loss      = DOMAIN_CRITERION(tgt_domain_logits, tgt_domain_labels)

            total_domain_loss = src_domain_loss + tgt_domain_loss
            total_loss        = src_class_loss + total_domain_loss

        # ── Backward pass with AMP scaler + gradient clipping ─────────────
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = _compute_grad_norm(model)   # measured AFTER unscale, BEFORE clip
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        # ── Batch accounting ───────────────────────────────────────────────
        n_src    = src_imgs.size(0)
        loss_sum += total_loss.item() * n_src
        preds     = (torch.sigmoid(src_class_logits) >= 0.5).long()
        correct_class += (preds == src_labels.long()).sum().item()
        total         += n_src

        # ── Log every batch to TrainingLogger ─────────────────────────────
        if logger is not None:
            logger.log_batch(
                total_loss=total_loss.item(),
                class_loss=src_class_loss.item(),
                domain_loss=total_domain_loss.item(),
                grad_norm=grad_norm,
                alpha=alpha,
                n_src=n_src,
                src_domain_logits=src_domain_logits.detach(),
                tgt_domain_logits=tgt_domain_logits.detach(),
            )

        pbar.set_postfix({
            "TotL": f"{total_loss.item():.3f}",
            "ClsL": f"{src_class_loss.item():.3f}",
            "DomL": f"{total_domain_loss.item():.3f}",
            "Acc":  f"{(preds == src_labels.long()).float().mean().item():.3f}",
            "GN":   f"{grad_norm:.3f}",
            "α":    f"{alpha:.3f}",
        })

    pbar.close()
    return loss_sum / max(total, 1), correct_class / max(total, 1)


# ── 7d. Evaluation ─────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(
    model:  DANN_ResNet18,
    loader: DataLoader,
    device: torch.device,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    """
    Inference-mode evaluation over a DataLoader.

    Returns:
        (mean_loss, accuracy, all_labels, all_probs)

    Note: model.train() is restored at the end so BatchNorm / Dropout
    are re-enabled for the subsequent training epoch.
    """
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_labels, all_probs   = [], []

    pbar = tqdm(loader, desc="  Evaluate", leave=False, ncols=80)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)                 # no alpha → class head only
            loss   = CLASS_CRITERION(logits, labels)

        probs     = torch.sigmoid(logits.float())
        loss_sum += loss.item() * imgs.size(0)
        correct  += ((probs >= 0.5).long() == labels.long()).sum().item()
        total    += imgs.size(0)
        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())

    model.train()   # restore training mode
    return (
        loss_sum / max(total, 1),
        correct  / max(total, 1),
        np.array(all_labels),
        np.array(all_probs),
    )


# ── 7e. Metric Suite ───────────────────────────────────────────────────────
def compute_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:
    """
    Computes the full clinical metric suite appropriate for medical imaging.

    Key metrics for brain tumour classification:
        - Sensitivity (Recall): Catches as many real tumours as possible. HIGH PRIORITY.
          A missed tumour (FN) has far worse clinical consequences than a false alarm.
        - Specificity: Avoids false positives — unnecessary follow-ups and patient anxiety.
        - AUC-ROC: Threshold-independent performance; most reliable single metric
          for imbalanced medical datasets.
        - F1: Balances precision and recall; useful for publication reporting.
    """
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),   # = Sensitivity
        "specificity": float(tn / (tn + fp + 1e-8)),
        "f1":          f1_score(labels, preds, zero_division=0),
        "auc_roc":     roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0,
    }


---
# 8. Cross-Validation Runner <a id='8'></a>

The CV runner now passes a `TrainingLogger` instance through every level:
`run_dann_cv` → `begin_fold` → loop → `begin_epoch` → `train_dann_epoch` → `end_epoch` → `end_fold` → `print_cv_summary`.


In [34]:
# ==========================================
# 8. DANN Cross-Validation Runner (Complete)
# ==========================================
# FLOW:
#   for each fold:
#       1. Split BRISC into train / val via StratifiedGroupKFold
#       2. Build DataLoaders (source train, source val, target cycle)
#       3. Instantiate fresh DANN_ResNet18 + AdamW + CosineAnnealingLR + GradScaler
#       4. for each epoch:
#             a. train_dann_epoch  → adversarial + class losses
#             b. evaluate          → val loss, acc, probs
#             c. compute_metrics   → sens / spec / AUC / F1
#             d. logger.end_epoch  → prints row + detects overfitting
#             e. Checkpoint best AUC model to Drive
#       5. logger.end_fold → confusion matrix + fold summary
#   logger.print_cv_summary → mean ± std across all folds
#   Save global best model state dict

import copy
import itertools

def run_dann_cv(brisc_records, mendeley_records, device, config, brisc_cache=None, mendeley_cache=None):
    """
    Runs stratified group k-fold cross-validation for the DANN pipeline.

    Args:
        brisc_records    : List of source-domain records from load_brisc().
        mendeley_records : List of target-domain records from load_mendeley().
        device           : torch.device (cuda or cpu).
        config           : CONFIG dict — single source of truth for all hyperparameters.
        brisc_cache      : Optional precomputed RAM cache for BRISC images.
        mendeley_cache   : Optional precomputed RAM cache for Mendeley images.

    Returns:
        (best_global_model, fold_results)
    """
    # ── Prepare arrays for StratifiedGroupKFold ────────────────────────────
    labels_arr = np.array([r["label"]      for r in brisc_records])
    groups_arr = np.array([r["patient_id"] for r in brisc_records])

    sgkf   = StratifiedGroupKFold(n_splits=config["n_folds"], shuffle=True, random_state=config["seed"])
    logger = TrainingLogger(out_dir=config["out_dir"], n_folds=config["n_folds"], total_epochs=config["num_epochs"])

    # ── Target DataLoader (shared across all folds — unlabelled) ──────────
    # Using augmented transforms on target as well (domain shift simulation).
    target_ds = BrainMRIDataset(mendeley_records, build_transforms(augment=True), cache=mendeley_cache)
    target_dl = DataLoader(
        target_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        drop_last=True,          # drop_last=True keeps batch sizes uniform for BatchNorm
        pin_memory=PIN_MEMORY,
    )

    fold_results       = []
    best_global_model  = None
    best_global_auc    = 0.0

    # ── Outer CV loop ──────────────────────────────────────────────────────
    for fold, (tr_idx, vl_idx) in enumerate(
        sgkf.split(brisc_records, labels_arr, groups=groups_arr), start=1
    ):
        train_recs = [brisc_records[i] for i in tr_idx]
        val_recs   = [brisc_records[i] for i in vl_idx]
        logger.begin_fold(fold=fold, n_train=len(train_recs), n_val=len(val_recs))

        # ── Source DataLoaders ─────────────────────────────────────────────
        train_ds = BrainMRIDataset(train_recs, build_transforms(augment=True),  cache=brisc_cache)
        val_ds   = BrainMRIDataset(val_recs,   build_transforms(augment=False), cache=brisc_cache)

        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True,
                              num_workers=config["num_workers"], drop_last=True, pin_memory=PIN_MEMORY)
        val_dl   = DataLoader(val_ds,   batch_size=config["batch_size"], shuffle=False,
                              num_workers=config["num_workers"], pin_memory=PIN_MEMORY)

        # ── Cycle target loader so it never exhausts ───────────────────────
        target_iter = itertools.cycle(target_dl)

        # ── Model, optimiser, scheduler, AMP scaler — fresh each fold ─────
        model     = DANN_ResNet18(pretrained=True).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=config["num_epochs"],
            eta_min=1e-6,   # floor LR so training never fully stops
        )
        scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

        best_fold_auc        = 0.0
        best_fold_model_state = None
        logger.print_epoch_header()

        # ── Inner epoch loop ───────────────────────────────────────────────
        for epoch in range(config["num_epochs"]):
            logger.begin_epoch(fold=fold, epoch=epoch)

            # --- 1. Train one epoch (adversarial + classification) ----------
            train_loss, train_acc = train_dann_epoch(
                model=model,
                source_loader=train_dl,
                target_iter=target_iter,
                optimizer=optimizer,
                scaler=scaler,
                device=device,
                current_epoch=epoch,
                total_epochs=config["num_epochs"],
                grad_clip_norm=config["grad_clip_norm"],
                logger=logger,
            )

            # --- 2. Validate (class head only, no GRL) ---------------------
            val_loss, val_acc, val_labels, val_probs = evaluate(model, val_dl, device)
            val_metrics = compute_metrics(val_labels, val_probs)

            # --- 3. Step LR scheduler AFTER validation ---------------------
            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]

            # --- 4. Checkpoint if best AUC so far in this fold -------------
            is_best = val_metrics["auc_roc"] > best_fold_auc
            if is_best:
                best_fold_auc         = val_metrics["auc_roc"]
                best_fold_model_state = copy.deepcopy(model.state_dict())
                save_path = os.path.join(config["out_dir"], f"best_model_fold{fold}.pth")
                torch.save(best_fold_model_state, save_path)
                logger.log_best_model(fold=fold, epoch=epoch+1, auc=best_fold_auc, save_path=save_path)

            # --- 5. Log epoch-level metrics to logger ----------------------
            logger.end_epoch(
                train_loss=train_loss,
                train_acc=train_acc,
                val_loss=val_loss,
                val_acc=val_acc,
                val_metrics=val_metrics,
                lr=current_lr,
                is_best=is_best,
            )

        # ── End of fold: restore best weights, final evaluation ────────────
        if best_fold_model_state is not None:
            model.load_state_dict(best_fold_model_state)
            log.info(f"  Fold {fold}: restored best-AUC model (AUC={best_fold_auc:.4f}) for final eval.")

        _, _, final_labels, final_probs = evaluate(model, val_dl, device)
        final_metrics = compute_metrics(final_labels, final_probs)
        logger.end_fold(metrics=final_metrics, val_labels=final_labels, val_probs=final_probs)

        fold_results.append({"fold": fold, "metrics": final_metrics})

        # Track global best across all folds
        if best_fold_auc > best_global_auc:
            best_global_auc   = best_fold_auc
            best_global_model = copy.deepcopy(model)

    # ── Final CV summary & save global best model ──────────────────────────
    logger.print_cv_summary()

    global_save_path = os.path.join(config["out_dir"], "best_model_global.pth")
    if best_global_model is not None:
        torch.save(best_global_model.state_dict(), global_save_path)
        log.info(f"  Global best model saved → {global_save_path}  (AUC={best_global_auc:.4f})")

    return best_global_model, fold_results


---
# 9. Master Execution <a id='9'></a>

In [35]:
# ==========================================
# 9. Master Execution
# ==========================================
def main():
    os.makedirs(CONFIG["out_dir"], exist_ok=True)
    set_seeds(CONFIG["seed"])

    brisc_path    = str(BRISC_PATH)
    mendeley_path = str(MENDELEY_PATH)

    log.info(f"Loading BRISC from: {brisc_path}")
    brisc_records = load_brisc(brisc_path, slices_per_patient=CONFIG["slices_per_patient"])

    log.info(f"Loading Mendeley from: {mendeley_path}")
    mendeley_records = load_mendeley(mendeley_path)

    if not brisc_records or not mendeley_records:
        log.error("One or more datasets failed to load. Check paths above.")
        return

    log.info("Building caches...")
    brisc_cache    = build_cache(brisc_records, cache_path=CONFIG["brisc_cache_path"])
    mendeley_cache = build_cache(mendeley_records, cache_path=CONFIG["mendeley_cache_path"])

    log.info(">>> Starting DANN Cross-Validation <<<")
    best_model, cv_results = run_dann_cv(
        brisc_records, mendeley_records, DEVICE, CONFIG,
        brisc_cache=brisc_cache, mendeley_cache=mendeley_cache,
    )

    # ── Section 10: External Validation ───────────────────────────────────
    # Runs automatically after CV completes. Uses the global best model
    # (highest AUC across all folds) on the 3 held-out external datasets.
    if best_model is not None:
        log.info("\n>>> Starting External Validation <<<")
        ext_results = run_external_validation(
            model=best_model,
            datasets=CLEANED_DATASETS,
            device=DEVICE,
            config=CONFIG,
        )
        print_external_summary_table(ext_results)
    else:
        log.error("best_model is None — CV may have failed. External validation skipped.")

if __name__ == "__main__":
    main()


NameError: name 'run_external_validation' is not defined

---
# 10. External Validation <a id='10'></a>

Runs the **global best model** (saved after CV) against all three held-out external datasets:

| Dataset | Variable | Clinical relevance |
|---------|----------|--------------------|
| Ayesha et al. | `Val_Ayesha` | Different scanner / centre |
| Alam et al.   | `Val_Alam`   | Different acquisition protocol |
| DeepPy        | `Val_DeepPy` | Curated deep-learning benchmark |

### What you get
- **In the notebook** — a printed metric table (sensitivity, specificity, AUC, F1, accuracy) per dataset, colour-coded in the log, plus an inline 3-panel figure with ROC curves and confusion matrices.
- **In your Drive** (`CONFIG["out_dir"]`) — `external_validation_results.json` (machine-readable) and `external_validation_plots.png` (the same figure, high-res).

> **Why run this separately from CV?**  
> The CV loop uses BRISC folds for train/val — those external datasets were never seen during training *or* hyperparameter selection, making them a true held-out generalisation test.  
> Mixing them into CV would inflate reported performance.


In [21]:
# ==========================================
# 10. External Validation
# ==========================================
# PURPOSE:
#   Evaluate the global best model (output of run_dann_cv) on the 3 external
#   held-out datasets. Results are printed in the notebook AND saved to Drive.
#
# REQUIRES:
#   - best_model  : DANN_ResNet18 instance returned by main() / run_dann_cv()
#   - CLEANED_DATASETS dict from Section 2 (paths already defined)
#   - evaluate() and compute_metrics() from Section 7

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve


def run_external_validation(
    model:    "DANN_ResNet18",
    datasets: dict,
    device:   "torch.device",
    config:   dict,
) -> dict:
    """
    Evaluates `model` on every dataset in `datasets` and writes results to Drive.

    Args:
        model    : Best global model returned by run_dann_cv().
        datasets : Dict of {name -> Path} — the 3 external val sets from CLEANED_DATASETS.
        device   : torch.device.
        config   : CONFIG dict (uses out_dir, batch_size, num_workers).

    Returns:
        Dict of {dataset_name -> metrics_dict} for downstream use.
    """
    # ── Which datasets to run (skip source + target used during training) ──
    EXTERNAL_KEYS = ["Val_Ayesha", "Val_Alam", "Val_DeepPy"]
    ext_datasets  = {k: datasets[k] for k in EXTERNAL_KEYS if k in datasets}

    if not ext_datasets:
        log.error("No external validation paths found in CLEANED_DATASETS. Check Section 2.")
        return {}

    val_transform = build_transforms(augment=False)   # deterministic only
    all_results   = {}
    all_labels_d  = {}
    all_probs_d   = {}

    log.info("\n" + "═" * 70)
    log.info("  EXTERNAL VALIDATION — Global Best Model")
    log.info("═" * 70)

    for name, path in ext_datasets.items():
        log.info(f"\n  ── {name}  ({path}) ──────────────────────────")

        # ── Load records ───────────────────────────────────────────────────
        records = load_mendeley(root_path=path, name=name)
        if not records:
            log.warning(f"  ⚠  No images found for {name}. Skipping.")
            continue

        # ── No caching for external sets — they run once, RAM is precious ──
        ds = BrainMRIDataset(records, val_transform, cache=None)
        dl = DataLoader(
            ds,
            batch_size=config["batch_size"],
            shuffle=False,
            num_workers=config["num_workers"],
            pin_memory=PIN_MEMORY,
        )

        # ── Evaluate ────────────────────────────────────────────────────────
        _, _, labels, probs = evaluate(model, dl, device)
        metrics = compute_metrics(labels, probs)
        all_results[name]  = metrics
        all_labels_d[name] = labels
        all_probs_d[name]  = probs

        # ── Print metric table inline ───────────────────────────────────────
        log.info(f"  {'Metric':<18}  {'Value':>8}")
        log.info(f"  {'─'*28}")
        for metric, value in metrics.items():
            flag = ""
            if metric == "recall"      and value < 0.80: flag = "  ⚠ LOW SENSITIVITY"
            if metric == "specificity" and value < 0.70: flag = "  ⚠ LOW SPECIFICITY"
            if metric == "auc_roc"     and value < 0.75: flag = "  ⚠ LOW AUC"
            log.info(f"  {metric:<18}  {value:>8.4f}{flag}")
        log.info(f"  {'─'*28}")
        log.info(f"  n_images = {len(records)},  n_tumor = {sum(r['label'] for r in records)},  n_no_tumor = {len(records) - sum(r['label'] for r in records)}")

    # ── Save JSON to Drive ──────────────────────────────────────────────────
    json_out = os.path.join(config["out_dir"], "external_validation_results.json")
    with open(json_out, "w") as f:
        json.dump(all_results, f, indent=2)
    log.info(f"\n  ✅ Results saved → {json_out}")

    # ── Plot: ROC curves + Confusion Matrices ──────────────────────────────
    n = len(all_results)
    if n == 0:
        log.warning("No results to plot.")
        return all_results

    fig = plt.figure(figsize=(7 * n, 10))
    fig.suptitle("External Validation — Global Best DANN Model", fontsize=16, fontweight="bold", y=1.01)
    gs  = gridspec.GridSpec(2, n, figure=fig, hspace=0.4, wspace=0.35)

    for col, (name, metrics) in enumerate(all_results.items()):
        labels = all_labels_d[name]
        probs  = all_probs_d[name]
        preds  = (probs >= 0.5).astype(int)

        # ── ROC curve (top row) ────────────────────────────────────────────
        ax_roc = fig.add_subplot(gs[0, col])
        fpr, tpr, _ = roc_curve(labels, probs)
        ax_roc.plot(fpr, tpr, lw=2, color="#4C9BE8", label=f"AUC = {metrics['auc_roc']:.3f}")
        ax_roc.plot([0, 1], [0, 1], "k--", lw=1)
        ax_roc.set_xlim([0, 1]); ax_roc.set_ylim([0, 1.02])
        ax_roc.set_xlabel("False Positive Rate"); ax_roc.set_ylabel("True Positive Rate")
        ax_roc.set_title(f"{name}\nROC Curve", fontsize=11)
        ax_roc.legend(loc="lower right", fontsize=10)
        # Annotate key metrics below the title
        ax_roc.text(0.05, 0.10,
            f"Sens={metrics['recall']:.3f}  Spec={metrics['specificity']:.3f}\n"
            f"F1={metrics['f1']:.3f}  Acc={metrics['accuracy']:.3f}",
            transform=ax_roc.transAxes, fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#F0F4FF", edgecolor="#4C9BE8"))

        # ── Confusion matrix (bottom row) ──────────────────────────────────
        ax_cm = fig.add_subplot(gs[1, col])
        ConfusionMatrixDisplay.from_predictions(
            labels, preds,
            labels=[0, 1],  # <--- ADD THIS EXACT LINE HERE
            display_labels=["No Tumor", "Tumor"],
            colorbar=False,
            ax=ax_cm,
            cmap="Blues",
        )
        ax_cm.set_title(f"{name}\nConfusion Matrix", fontsize=11)

    plt.tight_layout()

    # ── Save figure to Drive ───────────────────────────────────────────────
    plot_out = os.path.join(config["out_dir"], "external_validation_plots.png")
    plt.savefig(plot_out, dpi=150, bbox_inches="tight")
    log.info(f"  ✅ Plot saved     → {plot_out}")

    # ── Show inline in notebook ────────────────────────────────────────────
    plt.show()

    return all_results


# ── CROSS-VALIDATION SUMMARY TABLE ────────────────────────────────────────
def print_external_summary_table(ext_results: dict):
    """Prints a clean comparison table across all external datasets."""
    if not ext_results:
        return
    metrics_order = ["accuracy", "precision", "recall", "specificity", "f1", "auc_roc"]
    header = f"  {'Dataset':<18}" + "".join(f"  {m:>12}" for m in metrics_order)
    log.info("\n" + "═" * len(header))
    log.info("  EXTERNAL VALIDATION — SUMMARY TABLE")
    log.info("═" * len(header))
    log.info(header)
    log.info("  " + "─" * (len(header) - 2))
    for name, metrics in ext_results.items():
        row = f"  {name:<18}" + "".join(f"  {metrics.get(m, 0.0):>12.4f}" for m in metrics_order)
        log.info(row)
    log.info("═" * len(header))


In [22]:

# 1. Instantiate the model architecture (no need to download pretrained ImageNet weights again)
saved_model = DANN_ResNet18(pretrained=False).to(DEVICE)

# 2. Load the weights you already spent hours training from your Google Drive
model_path = os.path.join(CONFIG["out_dir"], "best_model_global.pth")
saved_model.load_state_dict(torch.load(model_path))
log.info(f"Successfully loaded trained model from {model_path}")

# 3. Run ONLY the external validation step
log.info("\n>>> Starting External Validation (Inference Only) <<<")
ext_results = run_external_validation(
    model=saved_model,
    datasets=CLEANED_DATASETS,
    device=DEVICE,
    config=CONFIG,
)

# 4. Print the final summary table
print_external_summary_table(ext_results)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1179: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/tmp/ipykernel_3292/3304643675.py:140: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [4]:
import os
from pathlib import Path

def get_detailed_counts(dataset_path):
    root = Path(dataset_path)
    if not root.exists():
        print(f"❌ Path not found: {root}")
        return

    print(f"\n{'='*50}")
    print(f"📊 Dataset: {root.name}")
    print(f"{'='*50}")

    # Dictionary to store counts: split -> class -> count
    counts = {}
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

    for dirpath, _, filenames in os.walk(root):
        # Count only image files
        images = [f for f in filenames if Path(f).suffix.lower() in img_exts]
        if not images:
            continue
            
        # Determine the subfolder structure (e.g., 'train/glioma_tumor')
        rel_path = Path(dirpath).relative_to(root)
        parts = rel_path.parts
        
        if len(parts) >= 2:
            split_name = parts[0] # Usually 'train', 'test', 'Training', or 'Testing'
            class_name = parts[1] # Usually 'glioma', 'pituitary', etc.
        elif len(parts) == 1:
            split_name = "mixed_split"
            class_name = parts[0]
        else:
            split_name = "root"
            class_name = "root"

        if split_name not in counts:
            counts[split_name] = {}
        if class_name not in counts[split_name]:
            counts[split_name][class_name] = 0
            
        counts[split_name][class_name] += len(images)

    # Print the formatted results
    if not counts:
        print("  [Empty Directory or No Images Found]")
        return

    for split, classes in sorted(counts.items()):
        print(f"\n📂 Split: [{split.upper()}]")
        total_in_split = 0
        for cls_name, count in sorted(classes.items()):
            print(f"   ├── {cls_name:<18} : {count}")
            total_in_split += count
        print(f"   └── {'TOTAL':<18} : {total_in_split}")

# Run this on your 3 external validation directories
get_detailed_counts(CLEANED_DATASETS["Source_BRISC"])
get_detailed_counts(CLEANED_DATASETS["Target_Mendeley"])
get_detailed_counts(CLEANED_DATASETS["Val_Ayesha"])
get_detailed_counts(CLEANED_DATASETS["Val_Alam"])
get_detailed_counts(CLEANED_DATASETS["Val_DeepPy"])


📊 Dataset: source

📂 Split: [TEST]
   ├── glioma             : 253
   ├── meningioma         : 294
   ├── no_tumor           : 138
   ├── pituitary          : 300
   └── TOTAL              : 985

📂 Split: [TRAIN]
   ├── glioma             : 1144
   ├── meningioma         : 1212
   ├── no_tumor           : 1056
   ├── pituitary          : 1433
   └── TOTAL              : 4845

📊 Dataset: target

📂 Split: [TEST]
   ├── glioma             : 307
   ├── meningioma         : 227
   ├── notumor            : 368
   ├── pituitary          : 42
   └── TOTAL              : 944

📂 Split: [TRAIN]
   ├── glioma             : 1372
   ├── meningioma         : 296
   ├── notumor            : 813
   ├── pituitary          : 666
   └── TOTAL              : 3147

📊 Dataset: External-validation-dataset-1

📂 Split: [TEST]
   ├── notumor            : 11
   └── TOTAL              : 11

📂 Split: [TRAIN]
   ├── glioma             : 39
   ├── meningioma         : 1
   ├── notumor            : 259
   └── TOTAL  